In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
import re
import math

from pathlib import Path

METHOD_LABELS = {
    "FO": "FO",
    "SO": "SO",
    "Rnd": "Bin",
    "CB": "C-bound",
    "exact": "SMV-Exact",
    "MC": "SMV-MC",
    "des_dir": "DIS-V",
    "des_rivasplata_dir": "DIS-R",
}

DATASET_LABELS = {"FASHION": "FASHION-MNIST"}

# the C-bound comes with its own certificate
BOUND_KEYS = {"CB": "cbound"}

DIRICHLET_METHODS = {"exact", "MC", "des_dir", "des_rivasplata_dir"}
DIRICHLET_PRIOR = 0.5
DIRICHLET_ORDER = 1.5
DIRICHLET_PRIOR_FALLBACKS = (0.1, 0.25, 0.5)

def find_available_prior(load_dir, path_for_prior):
    """Return the first available prior/path, without selecting on performance."""
    candidates = [DIRICHLET_PRIOR] + [
        prior for prior in DIRICHLET_PRIOR_FALLBACKS
        if not np.isclose(prior, DIRICHLET_PRIOR)
    ]
    for prior in candidates:
        try:
            return prior, next(load_dir.rglob(path_for_prior(prior)))
        except StopIteration:
            continue
    return None, None

PLOT_DIR = Path("../figures")
PLOT_DIR.mkdir(parents=True, exist_ok=True)


def make_binary_panel_axes(
    n_datasets, share_across_rows=True, separate_bottom_groups=False
):
    """Create two joined rows, with the shorter second row centred."""
    n_top = math.ceil(n_datasets / 2)
    n_bottom = n_datasets - n_top
    grid_cols = 2 * n_top

    fig = plt.figure(figsize=(2.15 * n_top, 4.8))
    grid = fig.add_gridspec(2, grid_cols, wspace=0.06, hspace=0.9)

    slots = [(0, 2 * i) for i in range(n_top)]
    bottom_offset = n_top - n_bottom
    slots += [(1, bottom_offset + 2 * i) for i in range(n_bottom)]

    axes = []
    shared_axis = None
    row_shared_axes = [None, None]
    bottom_shared_axes = [None, None]
    for row, start in slots:
        if share_across_rows:
            share_target = shared_axis
        elif row == 0 or not separate_bottom_groups:
            share_target = row_shared_axes[row]
        else:
            split = bottom_offset + 2 * math.ceil(n_bottom / 2)
            bottom_group = 0 if start < split else 1
            share_target = bottom_shared_axes[bottom_group]
        ax = fig.add_subplot(
            grid[row, start:start + 2],
            sharey=share_target,
        )
        if shared_axis is None:
            shared_axis = ax
        if row_shared_axes[row] is None:
            row_shared_axes[row] = ax
        if row == 1 and separate_bottom_groups:
            split = bottom_offset + 2 * math.ceil(n_bottom / 2)
            bottom_group = 0 if start < split else 1
            if bottom_shared_axes[bottom_group] is None:
                bottom_shared_axes[bottom_group] = ax
        axes.append(ax)

    return fig, axes, n_top


def split_datasets_by_scale(datasets, scales):
    """Create high, medium and low-amplitude scale groups."""
    low_scale = [d for d in datasets if d in {"SENSORLESS", "SHUTTLE"}]
    remaining = [d for d in datasets if d not in low_scale]

    # The three largest remaining amplitudes form the common high-scale group.
    high = set(sorted(remaining, key=scales.get, reverse=True)[:3])
    medium_scale = [d for d in remaining if d not in high]
    groups = [
        [d for d in remaining if d in high],
        low_scale,
        medium_scale,
    ]
    return [group for group in groups if group]


def split_time_groups(datasets, scales, n_high, n_low):
    """Group runtime panels by order of magnitude."""
    ranked = sorted(datasets, key=scales.get, reverse=True)
    high = set(ranked[:n_high])
    low = set(ranked[-n_low:])
    return [
        [d for d in datasets if d in high],
        [d for d in datasets if d in low],
        [d for d in datasets if d not in high and d not in low],
    ]


def make_grouped_panel_axes(groups):
    """Create joined axes inside each group and a gap between groups."""
    width_ratios = []
    for i, group in enumerate(groups):
        if i:
            width_ratios.append(0.35)
        width_ratios.extend([1.0] * len(group))

    fig = plt.figure(figsize=(2.0 * sum(map(len, groups)), 2.8))
    grid = fig.add_gridspec(
        1, len(width_ratios),
        width_ratios=width_ratios,
        wspace=0.06,
    )

    axes = {}
    column = 0
    for group_index, group in enumerate(groups):
        if group_index:
            column += 1
        shared_axis = None
        for d in group:
            ax = fig.add_subplot(grid[0, column], sharey=shared_axis)
            if shared_axis is None:
                shared_axis = ax
            axes[d] = ax
            column += 1

    return fig, axes


# Binary datasets

## Results loading

In [ ]:
from pathlib import Path
import numpy as np

M = 10
lr = 0.1
batch_size = 128

methods = ["FO", "SO", "Rnd", "CB", "exact", "MC", "des_dir", "des_rivasplata_dir"]
bound = "seeger"

datasets = ["MUSH", "TTT", "SVMGUIDE", "HABER", "PHIS", "CODRNA", "ADULT", "AUSTRALIAN", "SPLICE"]

priors_list = [0.1, 0.25, 0.5]
orders_list = [1.5, 2.5, 5.]
results = {}

for d in datasets:
    results[d] = {}
    for m in methods:
        M=10
        load_dir = Path(f"../results/{d}/{m}/{BOUND_KEYS.get(m, bound)}/optimize-bound=True/stochastic-bound=False/stumps-uniform/learnt_prior=False/M={M}/")
        results[d][m] = []

        if m == "uniform":
            path = next(load_dir.rglob(f"**/prior={priors_list[0]}/err-b.npy"))
            results[d][m].append(np.load(path, allow_pickle=True).item())

        elif m == "BayesianNB":
            path = next(Path(f"../results/all/{d}/{m}").rglob(f"**/err-b.npy"))
            results[d][m].append(np.load(path, allow_pickle=True).item())

        elif m in ["FO", "SO", "Rnd", "CB"]:
            p_prior = 1
            path = next(load_dir.rglob(f"**/prior={p_prior}/lr={lr}/batch-size={batch_size}/**/err-b.npy"))
            results[d][m].append(np.load(path, allow_pickle=True).item())

        elif m in ["exact", "MC", "des_rivasplata_dir"]:
            p_prior, path = find_available_prior(
                load_dir, lambda prior: f"**/prior={prior}/lr={lr}/batch-size={batch_size}/**/err-b.npy"
            )
            if path is not None:
                results[d][m].append({
                    "prior": p_prior,
                    "data": np.load(path, allow_pickle=True).item()
                })
                    
        elif m == "des_dir":
            method_priors = [DIRICHLET_PRIOR] + [prior for prior in DIRICHLET_PRIOR_FALLBACKS if not np.isclose(prior, DIRICHLET_PRIOR)]
            for p_prior in method_priors:
                for order in [DIRICHLET_ORDER]:
                    try:
                        path = next(load_dir.rglob(f"**/prior={p_prior}/lr={lr}/batch-size={batch_size}/order={order}/**/err-b.npy"))
                        results[d][m].append({
                            "prior": p_prior,
                            "order": order,
                            "data": np.load(path, allow_pickle=True).item()
                        })
                    except StopIteration:
                        pass


In [ ]:
methods = ["FO", "SO", "Rnd", "CB", "exact", "MC", "des_dir", "des_rivasplata_dir"]

results["TICTACTOE"] = results["TTT"]
results["HABERMAN"] = results["HABER"]
results["MUSHROOMS"] = results["MUSH"]
results["PHISHING"] = results["PHIS"]

datasets = ["MUSHROOMS", "TICTACTOE", "SVMGUIDE", "HABERMAN", "PHISHING", "CODRNA", "ADULT", "SPLICE", "AUSTRALIAN"]
#datasets = ["MUSHROOMS", "TICTACTOE", "SVMGUIDE", "HABERMAN", "PHISHING", "CODRNA", "SPLICE", "AUSTRALIAN"]


### Saving the results for best hyperparameters

In [ ]:
best_params = {}

for d in datasets:
    best_params[d] = {}
    for m in methods:
        entries = results[d][m]
        if m in DIRICHLET_METHODS:
            loaded_priors = [entry.get("prior") for entry in entries]
            if loaded_priors and any(not any(np.isclose(prior, candidate) for candidate in DIRICHLET_PRIOR_FALLBACKS) for prior in loaded_priors):
                raise ValueError(f"{d}/{m}: expected one of {DIRICHLET_PRIOR_FALLBACKS}, found {loaded_priors}.")
        if m == "des_dir":
            loaded_orders = [entry.get("order") for entry in entries]
            if loaded_orders and any(not np.isclose(order, DIRICHLET_ORDER) for order in loaded_orders):
                raise ValueError(f"{d}/{m}: expected order={DIRICHLET_ORDER}, found {loaded_orders}.")
        if not entries:
            raise FileNotFoundError(f"No result loaded for {d}/{m}. Run this method with prior={DIRICHLET_PRIOR} before generating the comparison.")

        # Methods with a single configuration
        if "data" not in entries[0]:
            data = entries[0]
            best_params[d][m] = {
                "hyperparams": {},
                "metrics": {
                    "bound": data[BOUND_KEYS.get(m, bound)],
                    "test-error": data["test-error"],
                    "train-error": data.get("train-error", None),
                    "time": data.get("time", None)
                }
            }

        else:
            # Select the configuration using the bound
            best_entry = entries[0] if m in DIRICHLET_METHODS else min(entries, key=lambda e: e["data"][BOUND_KEYS.get(m, bound)][0])
            best_params[d][m] = {
                "hyperparams": {
                    "prior": best_entry.get("prior"),
                    "order": best_entry.get("order"),
                    "var": best_entry.get("var"),
                    "M": best_entry.get("M"),
                },
                "metrics": {
                    "bound": best_entry["data"][BOUND_KEYS.get(m, bound)],
                    "test-error": best_entry["data"]["test-error"],
                    "train-error": best_entry["data"].get("train-error"),
                    "time": best_entry["data"].get("time"),
                    "KL": best_entry["data"].get("KL"),
                    "entropy": best_entry["data"].get("entropy"),
                }
            }

# Keep the binary results available after the multiclass section.
binary_best_params = best_params
binary_datasets = list(datasets)
binary_methods = list(methods)


## Binary LaTeX tables

The results are converted to percentages. The lowest displayed mean in each dataset column is typeset in bold, with ties determined after rounding to two decimal places.

In [ ]:
def make_binary_latex_table(
    best_params, datasets, methods, metric, caption, label, scale=100.0
):
    means = {
        d: {
            m: scale * best_params[d][m]["metrics"][metric][0]
            for m in methods
        }
        for d in datasets
    }
    displayed_min = {
        d: min(float(f"{means[d][m]:.2f}") for m in methods)
        for d in datasets
    }

    lines = [
        r"\begin{table*}[ht!]",
        r"\centering",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{2pt}",
        r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l" + "c" * len(datasets) + "}",
        r"\toprule",
        "Method & " + " & ".join(datasets) + r" \\",
        r"\midrule",
    ]

    for m in methods:
        cells = []
        for d in datasets:
            mean, std = best_params[d][m]["metrics"][metric]
            mean_pct, std_pct = scale * mean, scale * std
            displayed_mean = f"{mean_pct:.2f}"
            value = rf"{displayed_mean} \pm {std_pct:.2f}"

            if float(displayed_mean) == displayed_min[d]:
                value = rf"\mathbf{{{value}}}"

            cells.append(f"${value}$")

        method_label = METHOD_LABELS.get(m, m)
        lines.append(method_label + " & " + " & ".join(cells) + r" \\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}%",
        r"}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table*}",
    ])
    return "\n".join(lines)


In [ ]:
binary_bound_table = make_binary_latex_table(
    best_params,
    datasets,
    methods,
    metric="bound",
    caption="Mean bound values across binary datasets. Not learned priors.",
    label="tab:best_bounds_binary",
)

binary_test_error_table = make_binary_latex_table(
    best_params,
    datasets,
    methods,
    metric="test-error",
    caption=r"Mean test-error (\%) across binary datasets. Not learned priors.",
    label="tab:best_test_error_binary",
)

time_table_methods = list(methods)

binary_time_table = make_binary_latex_table(
    best_params,
    datasets,
    time_table_methods,
    metric="time",
    caption="Mean training time (s) across binary datasets.",
    label="tab:best_time_binary",
    scale=1.0,
)

binary_latex_tables = (
    binary_bound_table + "\n\n" + binary_test_error_table
    + "\n\n" + binary_time_table
)
print(binary_latex_tables)


## Plots

In [ ]:
def draw_bars(best_params, datasets, padding=0.05, margin=0.05):

    n_datasets = len(datasets)
    fig, axes, n_top = make_binary_panel_axes(n_datasets)
    x_pos = 1.05 * np.arange(len(methods))
    global_y_min = np.inf
    global_y_max = -np.inf

    for i, d in enumerate(datasets):
        ax = axes[i]

        bounds_mean = []
        bounds_std = []
        test_mean = []
        test_std = []
        x_labels = []

        for m in methods:
            metrics = best_params[d][m]["metrics"]

            b_mean, b_std = metrics["bound"]
            t_mean, t_std = metrics["test-error"]

            bounds_mean.append(b_mean)
            bounds_std.append(b_std)
            test_mean.append(t_mean)
            test_std.append(t_std)
            x_labels.append(METHOD_LABELS.get(m, m))

        bounds_mean = np.array(bounds_mean)
        bounds_std = np.array(bounds_std)
        test_mean = np.array(test_mean)
        test_std = np.array(test_std)

        ax.bar(x_pos, bounds_mean, yerr=bounds_std, alpha=0.8, hatch="//", label="bound")
        ax.bar(x_pos, test_mean, yerr=test_std, alpha=1, hatch="..", label="test error")

        y_min_val = min((bounds_mean - bounds_std).min(), (test_mean - test_std).min())
        y_max_val = max((bounds_mean + bounds_std).max(), (test_mean + test_std).max())
        global_y_min = min(global_y_min, y_min_val)
        global_y_max = max(global_y_max, y_max_val)

        ax.set_title(
            DATASET_LABELS.get(d, d), fontsize=15, pad=3, fontweight="bold"
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            x_labels, rotation=75, fontsize=11, fontweight="bold"
        )
        ax.tick_params(axis="y", labelsize=7)
        if i not in (0, n_top):
            ax.tick_params(axis="y", left=False, labelleft=False)

        if i == 0:
            ax.legend(fontsize=11, framealpha=1.0, edgecolor="black")

    y_range = global_y_max - global_y_min
    ymin = max(0, global_y_min - padding * y_range)
    ymax = global_y_max + margin * y_range
    axes[0].set_ylim(ymin, ymax)

    fig.subplots_adjust(left=0.055, right=0.995, bottom=0.10, top=0.96)
    fig.savefig(PLOT_DIR / "binary-best-bound-test-error.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
def draw_time(
    best_params, datasets, padding=0.05, margin=0.05, bottom_scale_gap=0.05
):

    n_datasets = len(datasets)
    time_methods = [m for m in methods if m != "Rnd"]
    fig, axes, n_top = make_binary_panel_axes(
        n_datasets, share_across_rows=False, separate_bottom_groups=True
    )
    x_pos = 1.05 * np.arange(len(time_methods))
    bottom_group_start = n_top + math.ceil((n_datasets - n_top) / 2)
    group_y_min = [np.inf, np.inf, np.inf]
    group_y_max = [-np.inf, -np.inf, -np.inf]

    for i, d in enumerate(datasets):
        ax = axes[i]

        time_mean = []
        time_std = []
        x_labels = []

        for m in time_methods:
            metrics = best_params[d][m]["metrics"]

            t = metrics.get("time", None)
            if t is None:
                # Method without a recorded runtime
                time_mean.append(0)
                time_std.append(0)
            else:
                mean, std = t
                time_mean.append(mean)
                time_std.append(std)

            x_labels.append(METHOD_LABELS.get(m, m))

        time_mean = np.array(time_mean)
        time_std = np.array(time_std)

        ax.bar(
            x_pos,
            time_mean,
            yerr=time_std,
            alpha=0.9,
            hatch="xx",
            label="time (s)"
        )

        # Robust y-axis limits
        y_min_val = (time_mean - time_std).min()
        y_max_val = (time_mean + time_std).max()
        group_index = 0 if i < n_top else (1 if i < bottom_group_start else 2)
        group_y_min[group_index] = min(group_y_min[group_index], y_min_val)
        group_y_max[group_index] = max(group_y_max[group_index], y_max_val)

        ax.set_title(
            DATASET_LABELS.get(d, d), fontsize=15, pad=3, fontweight="bold"
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            x_labels, rotation=75, fontsize=11, fontweight="bold"
        )
        ax.tick_params(axis="y", labelsize=7)
        if i not in (0, n_top, bottom_group_start):
            ax.tick_params(axis="y", left=False, labelleft=False)

        if i == 0:
            ax.legend(fontsize=11, framealpha=1.0, edgecolor="black")

    for group_index, axis_index in enumerate((0, n_top, bottom_group_start)):
        y_range = group_y_max[group_index] - group_y_min[group_index]
        ymin = max(0, group_y_min[group_index] - padding * y_range)
        ymax = group_y_max[group_index] + margin * y_range
        axes[axis_index].set_ylim(ymin, ymax)

    fig.subplots_adjust(left=0.055, right=0.995, bottom=0.10, top=0.96)

    # Separate the two scale groups on the second row while keeping
    # the panels inside each group visually joined.
    for ax in axes[n_top:bottom_group_start]:
        position = ax.get_position()
        ax.set_position([
            position.x0 - bottom_scale_gap / 2, position.y0,
            position.width, position.height,
        ])
    for ax in axes[bottom_group_start:]:
        position = ax.get_position()
        ax.set_position([
            position.x0 + bottom_scale_gap / 2, position.y0,
            position.width, position.height,
        ])
    fig.savefig(PLOT_DIR / "binary-best-time.pdf", bbox_inches="tight")
    plt.show()


In [ ]:
draw_bars(best_params, datasets)


In [ ]:
draw_time(best_params, datasets)


In [ ]:
def make_latex_table(
    best_params, datasets, methods, metric, learnt_priors, caption, label, scale=100.0
):
    means = {
        d: {
            m: scale * best_params[d][m][learnt_priors]["metrics"][metric][0]
            for m in methods
        }
        for d in datasets
    }
    displayed_min = {
        d: min(float(f"{means[d][m]:.2f}") for m in methods)
        for d in datasets
    }

    dataset_headers = [DATASET_LABELS.get(d, d) for d in datasets]
    lines = [
        r"\begin{table*}[ht!]",
        r"\centering",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{3pt}",
        r"\begin{tabular}{l" + "c" * len(datasets) + "}",
        r"\toprule",
        "Method & " + " & ".join(dataset_headers) + r" \\",
        r"\midrule",
    ]

    for m in methods:
        cells = []
        for d in datasets:
            mean, std = best_params[d][m][learnt_priors]["metrics"][metric]
            mean_pct, std_pct = scale * mean, scale * std
            displayed_mean = f"{mean_pct:.2f}"
            value = rf"{displayed_mean} \pm {std_pct:.2f}"

            if float(displayed_mean) == displayed_min[d]:
                value = rf"\mathbf{{{value}}}"

            cells.append(f"${value}$")

        method_label = METHOD_LABELS.get(m, m)
        lines.append(method_label + " & " + " & ".join(cells) + r" \\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table*}",
    ])
    return "\n".join(lines)

# bound_table = make_latex_table(
#     best_params,
#     datasets,
#     methods,
#     metric="bound",
#     learnt_priors=False,
#     caption="Mean bound values across datasets. Not learned priors.",
#     label="tab:best_bounds_updated",
# )

# test_error_table = make_latex_table(
#     best_params,
#     datasets,
#     methods,
#     metric="test-error",
#     learnt_priors=False,
#     caption=r"Mean test-error (\%) across datasets. Not learned priors.",
#     label="tab:best_test_error_updated",
# )

# time_table = make_latex_table(
#     best_params,
#     datasets,
#     methods,
#     metric="time",
#     learnt_priors=False,
#     caption="Mean training time (s) across multiclass datasets. Not learned priors.",
#     label="tab:best_time_multiclass",
#     scale=1.0,
# )

# latex_tables = bound_table + "\n\n" + test_error_table + "\n\n" + time_table
# print(latex_tables)


# Multiclass

## Results loading

In [ ]:
M = 100
lr = 0.1
batch_size = 1024

methods = ["FO", "SO", "Rnd", "CB", "exact", "MC", "des_dir", "des_rivasplata_dir"]
bound = "seeger"
max_depth = None

datasets = "PENDIGITS SHUTTLE SENSORLESS PROTEIN MNIST FASHION"
datasets = datasets.split(" ")

ns = [7494, 43500, 46807, 17766, 60000, 60000]
dims = [16, 9, 48, 357, 784, 784]

priors_list = [0.1, 0.25, 0.5]
orders_list = [DIRICHLET_ORDER]


results = {}
for d in datasets:
    
    results[d] = {}
    for m in methods:
        results[d][m] = []


for d in datasets:
    for m in methods:
        for learnt in [False]:
            load_dir = Path(f"../results/{d}/{m}/{BOUND_KEYS.get(m, bound)}/optimize-bound=True/stochastic-bound=False/rf//learnt_prior={learnt}/M={M}/")
            

            if m == "uniform":
                path = next(load_dir.rglob(f"**/prior={priors_list[0]}/err-b.npy"))
                results[d][m].append(np.load(path, allow_pickle=True).item())

            elif m == "BayesianNB":
                path = next(Path(f"../results/all/{d}/{m}").rglob(f"**/err-b.npy"))
                results[d][m].append(np.load(path, allow_pickle=True).item())

            elif m in ["FO", "SO", "Rnd", "CB"]:
                
                p_prior = 1
                path = next(load_dir.rglob(f"**/prior={p_prior}/lr={lr}/batch-size={batch_size}/**/err-b.npy"))
                results[d][m].append({"data" :np.load(path, allow_pickle=True).item(), "learnt_priors": learnt,})

            elif m in ["exact", "MC", "des_rivasplata_dir"]:
                method_priors = [DIRICHLET_PRIOR] + [prior for prior in DIRICHLET_PRIOR_FALLBACKS if not np.isclose(prior, DIRICHLET_PRIOR)]
                for p_prior in method_priors:
                    try:
                        path = next(load_dir.rglob(f"**/prior={p_prior}/lr={lr}/batch-size={batch_size}/**/err-b.npy"))
                        results[d][m].append({
                            "prior": p_prior,
                            "data": np.load(path, allow_pickle=True).item(),
                            "learnt_priors": learnt,
                        })
                    except StopIteration:
                        pass
            elif m == "des_dir":
                method_priors = [DIRICHLET_PRIOR] + [prior for prior in DIRICHLET_PRIOR_FALLBACKS if not np.isclose(prior, DIRICHLET_PRIOR)]
                for p_prior in method_priors:
                    for order in orders_list:
                        try:
                            path = next(load_dir.rglob(f"**/prior={p_prior}/lr={lr}/batch-size={batch_size}/order={order}/**/err-b.npy"))
                            results[d][m].append({
                                "prior": p_prior,
                                "order": order,
                                "data": np.load(path, allow_pickle=True).item(),
                                 "learnt_priors": learnt,
                            })
                        except StopIteration:
                            pass


In [ ]:
datasets = "PENDIGITS SHUTTLE SENSORLESS PROTEIN MNIST FASHION"
datasets = datasets.split(" ")


In [ ]:
best_params = {}

for d in datasets:
    best_params[d] = {}
    for m in methods:
        best_params[d][m] = {}
        entries = results[d][m]
        if m in DIRICHLET_METHODS:
            loaded_priors = [entry.get("prior") for entry in entries]
            if loaded_priors and any(not any(np.isclose(prior, candidate) for candidate in DIRICHLET_PRIOR_FALLBACKS) for prior in loaded_priors):
                raise ValueError(f"{d}/{m}: expected one of {DIRICHLET_PRIOR_FALLBACKS}, found {loaded_priors}.")
        if m == "des_dir":
            loaded_orders = [entry.get("order") for entry in entries]
            if loaded_orders and any(not np.isclose(order, DIRICHLET_ORDER) for order in loaded_orders):
                raise ValueError(f"{d}/{m}: expected order={DIRICHLET_ORDER}, found {loaded_orders}.")
        if not entries:
            raise FileNotFoundError(f"No result loaded for {d}/{m}. Run this method with prior={DIRICHLET_PRIOR} before generating the comparison.")
        
        # Single configuration without a learned-prior setting
        if "data" not in entries[0]:
           
            data = entries[0]
            best_params[d][m][None] = {
                "hyperparams": {},
                "metrics": {
                    "bound": data[BOUND_KEYS.get(m, bound)],
                    "test-error": data["test-error"],
                    "train-error": data.get("train-error"),
                    "time": data.get("time"),
                    "KL": data.get("KL"),
                    "entropy": data.get("entropy"),
                }
            }
                    
        else:
            # Group entries by learned-prior setting
            groups = {}
            for e in entries:
                lp = e.get("learnt_priors", None)  # True / False / None
                if lp not in groups:
                    groups[lp] = []
                groups[lp].append(e)

            # Select each learned-prior group independently
            for lp, group_entries in groups.items():

                best_entry = group_entries[0] if m in DIRICHLET_METHODS else min(group_entries, key=lambda e: e["data"][BOUND_KEYS.get(m, bound)][0])
                best_params[d][m][lp] = {
                    "hyperparams": {
                        "prior": best_entry.get("prior"),
                        "order": best_entry.get("order"),
                        "var": best_entry.get("var"),
                        "M": best_entry.get("M"),
                        "learnt_priors": lp
                    },
                    "metrics": {
                        "bound": best_entry["data"][BOUND_KEYS.get(m, bound)],
                        "test-error": best_entry["data"]["test-error"],
                        "train-error": best_entry["data"].get("train-error"),
                        "time": best_entry["data"].get("time"),
                        "KL": best_entry["data"].get("KL"),
                        "entropy": best_entry["data"].get("entropy"),
                    }
                }

# Names used by the compact two-row figures below.
multiclass_best_params = best_params
multiclass_datasets = list(datasets)
multiclass_methods = list(methods)


## Plots

In [ ]:
def draw_bars(best_params, datasets, learnt_priors=None, padding=0.05, margin=0.05):
    """
    learnt_priors:
        True  -> utilise les configs learnt_prior=True
        False -> utilise les configs learnt_prior=False
        None  -> methods without this parameter
    """

    n_datasets = len(datasets)

    dataset_scales = {}
    for d in datasets:
        values = []
        for m in methods:
            if learnt_priors in best_params[d][m]:
                entry = best_params[d][m][learnt_priors]
            elif None in best_params[d][m]:
                entry = best_params[d][m][None]
            else:
                continue
            metrics = entry["metrics"]
            b_mean, b_std = metrics["bound"]
            t_mean, t_std = metrics["test-error"]
            values.extend([b_mean + b_std, t_mean + t_std])
        dataset_scales[d] = max(values)

    groups = split_datasets_by_scale(datasets, dataset_scales)
    dataset_group = {d: i for i, group in enumerate(groups) for d in group}
    plot_datasets = [d for group in groups for d in group]
    fig, axes = make_grouped_panel_axes(groups)
    x_pos = 1.05 * np.arange(len(methods))
    group_y_min = [np.inf] * len(groups)
    group_y_max = [-np.inf] * len(groups)

    for d in plot_datasets:
        ax = axes[d]
        group_index = dataset_group[d]

        bounds_mean = []
        bounds_std = []
        test_mean = []
        test_std = []
        x_labels = []

        for m in methods:
            # Retrieve the selected learned-prior setting
            if learnt_priors in best_params[d][m]:
                entry = best_params[d][m][learnt_priors]
            elif None in best_params[d][m]:
                entry = best_params[d][m][None]
            else:
                # No data for this learned-prior setting
                bounds_mean.append(np.nan)
                bounds_std.append(0)
                test_mean.append(np.nan)
                test_std.append(0)
                x_labels.append(METHOD_LABELS.get(m, m))
                continue

            metrics = entry["metrics"]

            b_mean, b_std = metrics["bound"]
            t_mean, t_std = metrics["test-error"]

            bounds_mean.append(b_mean)
            bounds_std.append(b_std)
            test_mean.append(t_mean)
            test_std.append(t_std)
            x_labels.append(METHOD_LABELS.get(m, m))

        bounds_mean = np.array(bounds_mean, dtype=float)
        bounds_std = np.array(bounds_std, dtype=float)
        test_mean = np.array(test_mean, dtype=float)
        test_std = np.array(test_std, dtype=float)

        ax.bar(x_pos, bounds_mean, yerr=bounds_std, alpha=0.8, hatch="//", label="bound")
        ax.bar(x_pos, test_mean, yerr=test_std, alpha=1, hatch="..", label="test error")

        # Robust y-axis limits
        finite_vals = np.isfinite(np.concatenate([
            bounds_mean - bounds_std,
            bounds_mean + bounds_std,
            test_mean - test_std,
            test_mean + test_std
        ]))

        all_vals = np.concatenate([
            bounds_mean - bounds_std,
            bounds_mean + bounds_std,
            test_mean - test_std,
            test_mean + test_std
        ])

        y_min_val = np.min(all_vals[finite_vals])
        y_max_val = np.max(all_vals[finite_vals])
        group_y_min[group_index] = min(group_y_min[group_index], y_min_val)
        group_y_max[group_index] = max(group_y_max[group_index], y_max_val)

        ax.set_title(
            DATASET_LABELS.get(d, d), fontsize=11, pad=3, fontweight="bold"
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            x_labels, rotation=75, fontsize=11, fontweight="bold"
        )
        ax.tick_params(axis="y", labelsize=7)
        if d != groups[group_index][0]:
            ax.tick_params(axis="y", left=False, labelleft=False)

    for group_index, group in enumerate(groups):
        y_range = group_y_max[group_index] - group_y_min[group_index]
        ymin = max(0, group_y_min[group_index] - padding * y_range)
        ymax = group_y_max[group_index] + margin * y_range
        axes[group[0]].set_ylim(ymin, ymax)

    axes[groups[0][-1]].legend(fontsize=11, framealpha=1.0, edgecolor="black")

    fig.subplots_adjust(left=0.06, right=0.995, bottom=0.27, top=0.91)
    fig.savefig(
        PLOT_DIR / f"multiclass-best-bound-test-error-learnt-prior={learnt_priors}.pdf",
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
def draw_time(best_params, datasets, learnt_priors=None, padding=0.05, margin=0.05):
    """
    learnt_priors:
        True  -> configs learnt_prior=True
        False -> configs learnt_prior=False
        None  -> methods without this parameter
    """

    n_datasets = len(datasets)
    time_methods = [m for m in methods if m != "Rnd"]

    dataset_scales = {}
    for d in datasets:
        values = []
        for m in time_methods:
            if learnt_priors in best_params[d][m]:
                entry = best_params[d][m][learnt_priors]
            elif None in best_params[d][m]:
                entry = best_params[d][m][None]
            else:
                continue
            time_metric = entry["metrics"].get("time")
            if time_metric is not None:
                mean, std = time_metric
                values.append(mean + std)
        dataset_scales[d] = max(values, default=0.0)

    groups = split_time_groups(datasets, dataset_scales, n_high=2, n_low=1)
    dataset_group = {d: i for i, group in enumerate(groups) for d in group}
    plot_datasets = [d for group in groups for d in group]
    fig, axes = make_grouped_panel_axes(groups)
    x_pos = 1.05 * np.arange(len(time_methods))
    group_y_min = [np.inf] * len(groups)
    group_y_max = [-np.inf] * len(groups)

    for d in plot_datasets:
        ax = axes[d]
        group_index = dataset_group[d]

        time_mean = []
        time_std = []
        x_labels = []

        for m in time_methods:
            # Select the requested learned-prior setting
            if learnt_priors in best_params[d][m]:
                entry = best_params[d][m][learnt_priors]
            elif None in best_params[d][m]:
                entry = best_params[d][m][None]
            else:
                # No data for this learned-prior setting
                time_mean.append(np.nan)
                time_std.append(0)
                x_labels.append(METHOD_LABELS.get(m, m))
                continue

            metrics = entry["metrics"]

            t = metrics.get("time", None)
            if t is None:
                time_mean.append(0)
                time_std.append(0)
            else:
                mean, std = t
                time_mean.append(mean)
                time_std.append(std)

            x_labels.append(METHOD_LABELS.get(m, m))

        time_mean = np.array(time_mean, dtype=float)
        time_std = np.array(time_std, dtype=float)

        ax.bar(
            x_pos,
            time_mean,
            yerr=time_std,
            alpha=0.9,
            hatch="xx",
            label="time (s)"
        )

        # Robust NaN-safe y-axis limits
        all_vals = np.concatenate([
            time_mean - time_std,
            time_mean + time_std
        ])

        finite = np.isfinite(all_vals)
        y_min_val = np.min(all_vals[finite])
        y_max_val = np.max(all_vals[finite])
        group_y_min[group_index] = min(group_y_min[group_index], y_min_val)
        group_y_max[group_index] = max(group_y_max[group_index], y_max_val)

        ax.set_title(
            DATASET_LABELS.get(d, d), fontsize=11, pad=3, fontweight="bold"
        )
        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            x_labels, rotation=75, fontsize=11, fontweight="bold"
        )
        ax.tick_params(axis="y", labelsize=7)
        if d != groups[group_index][0]:
            ax.tick_params(axis="y", left=False, labelleft=False)

    for group_index, group in enumerate(groups):
        y_range = group_y_max[group_index] - group_y_min[group_index]
        ymin = max(0, group_y_min[group_index] - padding * y_range)
        ymax = group_y_max[group_index] + margin * y_range
        axes[group[0]].set_ylim(ymin, ymax)

    axes[groups[0][-1]].legend(fontsize=11, framealpha=1.0, edgecolor="black")

    fig.subplots_adjust(left=0.06, right=0.995, bottom=0.27, top=0.91)
    fig.savefig(
        PLOT_DIR / f"multiclass-best-time-learnt-prior={learnt_priors}.pdf",
        bbox_inches="tight",
    )
    plt.show()


In [ ]:
draw_bars(best_params, datasets, learnt_priors=False)
#draw_bars(best_params, datasets, learnt_priors=True)


In [ ]:
draw_time(best_params, datasets, learnt_priors=False)
#draw_time(best_params, datasets, learnt_priors=True)


## LaTeX tables

The values are converted to percentages and the lowest displayed mean in each dataset column is typeset in bold. Ties are determined after rounding to two decimal places.

In [ ]:
def make_latex_table(
    best_params, datasets, methods, metric, learnt_priors, caption, label, scale=100.0
):
    means = {
        d: {
            m: scale * best_params[d][m][learnt_priors]["metrics"][metric][0]
            for m in methods
        }
        for d in datasets
    }
    displayed_min = {
        d: min(float(f"{means[d][m]:.2f}") for m in methods)
        for d in datasets
    }

    dataset_headers = [DATASET_LABELS.get(d, d) for d in datasets]
    lines = [
        r"\begin{table*}[ht!]",
        r"\centering",
        r"\scriptsize",
        r"\setlength{\tabcolsep}{3pt}",
        r"\begin{tabular}{l" + "c" * len(datasets) + "}",
        r"\toprule",
        "Method & " + " & ".join(dataset_headers) + r" \\",
        r"\midrule",
    ]

    for m in methods:
        cells = []
        for d in datasets:
            mean, std = best_params[d][m][learnt_priors]["metrics"][metric]
            mean_pct, std_pct = scale * mean, scale * std
            displayed_mean = f"{mean_pct:.2f}"
            value = rf"{displayed_mean} \pm {std_pct:.2f}"

            if float(displayed_mean) == displayed_min[d]:
                value = rf"\mathbf{{{value}}}"

            cells.append(f"${value}$")

        method_label = METHOD_LABELS.get(m, m)
        lines.append(method_label + " & " + " & ".join(cells) + r" \\")

    lines.extend([
        r"\bottomrule",
        r"\end{tabular}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table*}",
    ])
    return "\n".join(lines)


bound_table = make_latex_table(
    best_params,
    datasets,
    methods,
    metric="bound",
    learnt_priors=False,
    caption="Mean bound values across datasets. Not learned priors.",
    label="tab:best_bounds_updated",
)

test_error_table = make_latex_table(
    best_params,
    datasets,
    methods,
    metric="test-error",
    learnt_priors=False,
    caption=r"Mean test-error (\%) across datasets. Not learned priors.",
    label="tab:best_test_error_updated",
)

time_table_methods = list(methods)

time_table = make_latex_table(
    best_params,
    datasets,
    time_table_methods,
    metric="time",
    learnt_priors=False,
    caption="Mean training time (s) across multiclass datasets. Not learned priors.",
    label="tab:best_time_multiclass",
    scale=1.0,
)

latex_tables = bound_table + "\n\n" + test_error_table + "\n\n" + time_table
print(latex_tables)


## Compact two-row bar plots

Each figure uses one row for binary datasets and one row for multiclass datasets. Scale groups, vertical margins and method labels follow the preceding bar plots.

In [ ]:
from matplotlib.patches import Patch

COMPACT_BOUND_COLOR = "#4C78A8"
COMPACT_TEST_COLOR = "#F28E2B"
COMPACT_TIME_COLOR = "#4C9F8A"
COMPACT_METHOD_SPACING = 1.14
COMPACT_PANEL_WSPACE = 0.06
COMPACT_TITLE_SIZE = 34
COMPACT_METHOD_SIZE = 27
COMPACT_TICK_SIZE = 18
COMPACT_ROW_LABEL_SIZE = 25
COMPACT_LEGEND_SIZE = 25
COMPACT_ERROR_STYLE = {
    "ecolor": "0.15", "elinewidth": 0.8, "capsize": 1.8, "capthick": 0.8,
}


def compact_multiclass_entry(best_params, dataset, method, learnt_prior):
    regimes = best_params[dataset][method]
    if learnt_prior in regimes:
        return regimes[learnt_prior]
    if None in regimes:
        return regimes[None]
    return None


def make_compact_grouped_row(fig, subplot_spec, groups, gap=0.35):
    """One row of joined panels, with a visible gap between scale groups."""
    width_ratios = []
    for group_index, group in enumerate(groups):
        if group_index:
            width_ratios.append(gap)
        width_ratios.extend([1.0] * len(group))

    grid = subplot_spec.subgridspec(
        1, len(width_ratios), width_ratios=width_ratios,
        wspace=COMPACT_PANEL_WSPACE,
    )
    axes = {}
    column = 0
    for group_index, group in enumerate(groups):
        if group_index:
            column += 1
        shared_axis = None
        for dataset in group:
            ax = fig.add_subplot(grid[0, column], sharey=shared_axis)
            if shared_axis is None:
                shared_axis = ax
            axes[dataset] = ax
            column += 1
    return axes


def style_compact_axis(ax, dataset, methods, show_yaxis):
    x = COMPACT_METHOD_SPACING * np.arange(len(methods))
    ax.set_title(
        DATASET_LABELS.get(dataset, dataset),
        fontsize=COMPACT_TITLE_SIZE, pad=15,
    )
    ax.set_xlim(-0.57 * COMPACT_METHOD_SPACING,
                (len(methods) - 0.43) * COMPACT_METHOD_SPACING)
    ax.set_xticks(x)
    ax.set_xticklabels(
        [METHOD_LABELS.get(method, method) for method in methods],
        rotation=68, ha="right", rotation_mode="anchor",
        fontsize=COMPACT_METHOD_SIZE,
    )
    ax.tick_params(axis="x", length=0, pad=4)
    ax.tick_params(
        axis="y", labelsize=COMPACT_TICK_SIZE, length=4, pad=3,
    )
    if not show_yaxis:
        ax.tick_params(axis="y", left=False, labelleft=False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", color="0.88", linewidth=0.55, zorder=0)
    ax.set_axisbelow(True)


def compact_performance_values(best_params, dataset, methods, entry_getter):
    bound_mean, bound_std, test_mean, test_std = [], [], [], []
    for method in methods:
        entry = entry_getter(best_params, dataset, method)
        if entry is None:
            bound_mean.append(np.nan)
            bound_std.append(0.0)
            test_mean.append(np.nan)
            test_std.append(0.0)
            continue
        b_mean, b_std = entry["metrics"]["bound"]
        t_mean, t_std = entry["metrics"]["test-error"]
        bound_mean.append(b_mean)
        bound_std.append(b_std)
        test_mean.append(t_mean)
        test_std.append(t_std)
    return tuple(
        np.asarray(values, dtype=float)
        for values in (bound_mean, bound_std, test_mean, test_std)
    )


def performance_scale_groups(best_params, groups, methods, entry_getter):
    limits = []
    for group in groups:
        group_min, group_max = np.inf, -np.inf
        for dataset in group:
            b_mean, b_std, t_mean, t_std = compact_performance_values(
                best_params, dataset, methods, entry_getter,
            )
            values = np.concatenate([
                b_mean - b_std, b_mean + b_std,
                t_mean - t_std, t_mean + t_std,
            ])
            values = values[np.isfinite(values)]
            group_min = min(group_min, values.min())
            group_max = max(group_max, values.max())
        value_range = group_max - group_min
        limits.append((
            max(0, group_min - 0.05 * value_range),
            group_max + 0.05 * value_range,
        ))
    return limits


def draw_compact_performance_row(
    axes, best_params, groups, methods, entry_getter,
):
    limits = performance_scale_groups(
        best_params, groups, methods, entry_getter,
    )
    x = COMPACT_METHOD_SPACING * np.arange(len(methods))
    for group, ylim in zip(groups, limits):
        for dataset_index, dataset in enumerate(group):
            ax = axes[dataset]
            b_mean, b_std, t_mean, t_std = compact_performance_values(
                best_params, dataset, methods, entry_getter,
            )
            ax.bar(
                x, b_mean, width=0.78, yerr=b_std,
                color=COMPACT_BOUND_COLOR, edgecolor="0.15", linewidth=0.4,
                hatch="//", error_kw=COMPACT_ERROR_STYLE, zorder=2,
            )
            ax.bar(
                x, t_mean, width=0.78, yerr=t_std,
                color=COMPACT_TEST_COLOR, edgecolor="0.15", linewidth=0.4,
                hatch="..", error_kw=COMPACT_ERROR_STYLE, zorder=3,
            )
            style_compact_axis(
                ax, dataset, methods, show_yaxis=(dataset_index == 0),
            )
        axes[group[0]].set_ylim(*ylim)


def draw_compact_performance(learnt_prior=False, figsize=(36.0, 13.5)):
    binary_plot_methods = list(binary_methods)
    multiclass_plot_methods = list(multiclass_methods)
    binary_groups = [binary_datasets]
    binary_entry = lambda params, dataset, method: params[dataset][method]
    multiclass_entry = lambda params, dataset, method: compact_multiclass_entry(
        params, dataset, method, learnt_prior,
    )

    dataset_scales = {}
    for dataset in multiclass_datasets:
        values = compact_performance_values(
            multiclass_best_params, dataset, multiclass_plot_methods,
            multiclass_entry,
        )
        dataset_scales[dataset] = max(
            np.nanmax(values[0] + values[1]),
            np.nanmax(values[2] + values[3]),
        )
    multiclass_groups = split_datasets_by_scale(
        multiclass_datasets, dataset_scales,
    )

    fig = plt.figure(figsize=figsize)
    outer = fig.add_gridspec(2, 1, hspace=1.02)
    binary_axes = make_compact_grouped_row(fig, outer[0], binary_groups)
    multiclass_axes = make_compact_grouped_row(
        fig, outer[1], multiclass_groups,
    )
    draw_compact_performance_row(
        binary_axes, binary_best_params, binary_groups,
        binary_plot_methods, binary_entry,
    )
    draw_compact_performance_row(
        multiclass_axes, multiclass_best_params, multiclass_groups,
        multiclass_plot_methods, multiclass_entry,
    )
    binary_axes[binary_groups[0][0]].set_ylabel(
        "Binary", fontsize=COMPACT_ROW_LABEL_SIZE, fontweight="bold",
    )
    multiclass_axes[multiclass_groups[0][0]].set_ylabel(
        "Multiclass", fontsize=COMPACT_ROW_LABEL_SIZE, fontweight="bold",
    )
    binary_axes[binary_groups[0][0]].legend(
        handles=[
            Patch(facecolor=COMPACT_BOUND_COLOR, edgecolor="0.15", hatch="//", label="Bound"),
            Patch(facecolor=COMPACT_TEST_COLOR, edgecolor="0.15", hatch="..", label="Test error"),
        ],
        loc="upper left", ncol=1, frameon=True, framealpha=0.9,
        fontsize=COMPACT_LEGEND_SIZE,
        bbox_to_anchor=(0.01, 0.99), borderaxespad=0.0,
    )
    fig.subplots_adjust(left=0.05, right=0.995, bottom=0.18, top=0.96)
    output_path = PLOT_DIR / "compact-bound-test-error.pdf"
    fig.savefig(output_path, bbox_inches="tight")
    return fig, output_path


def compact_time_values(best_params, dataset, methods, entry_getter):
    means, stds = [], []
    for method in methods:
        entry = entry_getter(best_params, dataset, method)
        metric = None if entry is None else entry["metrics"].get("time")
        if metric is None:
            means.append(0.0)
            stds.append(0.0)
        else:
            mean, std = metric
            means.append(mean)
            stds.append(std)
    return np.asarray(means), np.asarray(stds)


def draw_compact_time_row(axes, best_params, groups, methods, entry_getter):
    x = COMPACT_METHOD_SPACING * np.arange(len(methods))
    for group in groups:
        group_min, group_max = np.inf, -np.inf
        for dataset in group:
            means, stds = compact_time_values(
                best_params, dataset, methods, entry_getter,
            )
            group_min = min(group_min, (means - stds).min())
            group_max = max(group_max, (means + stds).max())
        value_range = group_max - group_min
        ylim = (
            max(0, group_min - 0.05 * value_range),
            group_max + 0.05 * value_range,
        )
        for dataset_index, dataset in enumerate(group):
            ax = axes[dataset]
            means, stds = compact_time_values(
                best_params, dataset, methods, entry_getter,
            )
            ax.bar(
                x, means, width=0.78, yerr=stds,
                color=COMPACT_TIME_COLOR, edgecolor="0.15", linewidth=0.4,
                hatch="xx", error_kw=COMPACT_ERROR_STYLE, zorder=2,
            )
            style_compact_axis(
                ax, dataset, methods, show_yaxis=(dataset_index == 0),
            )
        axes[group[0]].set_ylim(*ylim)


def draw_compact_time(learnt_prior=False, figsize=(36.0, 13.5)):
    binary_plot_methods = [
        method for method in binary_methods
        if method not in {"FO", "SO", "Rnd"}
    ]
    multiclass_plot_methods = [
        method for method in multiclass_methods
        if method not in {"FO", "SO", "Rnd"}
    ]
    n_top = math.ceil(len(binary_datasets) / 2)
    bottom_group_start = n_top + math.ceil((len(binary_datasets) - n_top) / 2)
    binary_groups = [
        binary_datasets[:n_top],
        binary_datasets[n_top:bottom_group_start],
        binary_datasets[bottom_group_start:],
    ]
    binary_groups = [group for group in binary_groups if group]
    binary_entry = lambda params, dataset, method: params[dataset][method]
    multiclass_entry = lambda params, dataset, method: compact_multiclass_entry(
        params, dataset, method, learnt_prior,
    )

    dataset_scales = {}
    for dataset in multiclass_datasets:
        means, stds = compact_time_values(
            multiclass_best_params, dataset, multiclass_plot_methods,
            multiclass_entry,
        )
        dataset_scales[dataset] = np.max(means + stds)
    multiclass_groups = split_time_groups(
        multiclass_datasets, dataset_scales, n_high=2, n_low=1,
    )

    fig = plt.figure(figsize=figsize)
    outer = fig.add_gridspec(2, 1, hspace=1.02)
    binary_axes = make_compact_grouped_row(fig, outer[0], binary_groups)
    multiclass_axes = make_compact_grouped_row(
        fig, outer[1], multiclass_groups,
    )
    draw_compact_time_row(
        binary_axes, binary_best_params, binary_groups,
        binary_plot_methods, binary_entry,
    )
    draw_compact_time_row(
        multiclass_axes, multiclass_best_params, multiclass_groups,
        multiclass_plot_methods, multiclass_entry,
    )
    binary_axes[binary_groups[0][0]].set_ylabel(
        "Binary", fontsize=COMPACT_ROW_LABEL_SIZE,
        fontweight="bold",
        labelpad=15,
    )
    multiclass_axes[multiclass_groups[0][0]].set_ylabel(
        "Multiclass", fontsize=COMPACT_ROW_LABEL_SIZE,
        fontweight="bold",
        labelpad=15,
    )
    binary_axes[binary_groups[0][0]].legend(
        handles=[Patch(
            facecolor=COMPACT_TIME_COLOR, edgecolor="0.15",
            hatch="xx", label="Runtime (s)",
        )],
        loc="upper left", frameon=True, framealpha=0.9,
        fontsize=COMPACT_LEGEND_SIZE,
        bbox_to_anchor=(0.01, 0.99), borderaxespad=0.0,
    )
    fig.subplots_adjust(left=0.05, right=0.995, bottom=0.18, top=0.96)
    output_path = PLOT_DIR / "compact-time.pdf"
    fig.savefig(output_path, bbox_inches="tight")
    return fig, output_path


compact_performance_fig, compact_performance_path = draw_compact_performance(
    learnt_prior=False,
)
plt.show()

compact_time_fig, compact_time_path = draw_compact_time(
    learnt_prior=False,
)
plt.show()
